# Uniform excitation: W and G

This notebook uses synthetic data only. Select demo or study below. The demo does not reproduce the paper results. Each section has short cells so you can inspect the intermediate results.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import importlib
import src.simulation as simulation_module
importlib.reload(simulation_module)  # Refresh progress display after an update.
from src.simulation import make_design, run_bank
from src.plotting import heatmaps, validation_figures
import pandas as pd

CONFIG_NAME = 'demo'  # Change to 'study' to use the original network.
CONFIG = ROOT / 'configs' / (CONFIG_NAME + '.json')
MODE = 'uniform'
OUTPUT = ROOT / 'outputs' / CONFIG_NAME / MODE

## 1. Explore the model
Use a 10 x 10 grid (100 simulations). Increase the square count for finer maps. Full-length simulations are computationally expensive. The mean and variance are temporal statistics computed in each region and then averaged across regions, before z-scoring. The same W is assigned to all regions.

In [ ]:
N_GRID = 100
RUN_GRID = False  # Set True to generate simulations; False loads saved results.
WORKERS = 1       # Use the command-line script for large parallel jobs.
GRID_FOLDER = OUTPUT / 'exploration'
if RUN_GRID:
    design = make_design(MODE, count=N_GRID, grid=True, fixed_G=1.89)
    grid_data = run_bank(design, CONFIG, GRID_FOLDER, workers=WORKERS)
else:
    if not (GRID_FOLDER / 'simulations.csv').exists():
        raise FileNotFoundError('Set RUN_GRID=True for the first run.')
    grid_data = pd.read_csv(GRID_FOLDER / 'simulations.csv')
grid_data.head()

In [ ]:
heatmaps(grid_data, 'w', 'G', GRID_FOLDER / 'figures')

## 2. Generate or import the inference bank
This is a separate uniform random sample of the prior. Do not use the fixed-G grid to infer G. 4,000 cases are an example; set 20,000 for a larger bank. Import an existing bank by setting BANK_FOLDER to its directory. Keep its provenance with it.

In [ ]:
N_BANK = 4000
RUN_BANK = False
BUNDLED_BANK = ROOT / 'data' / 'demo_uniform4000'
BANK_FOLDER = BUNDLED_BANK if CONFIG_NAME == 'demo' and not RUN_BANK else OUTPUT / 'inference_bank'
if RUN_BANK:
    design = make_design(MODE, count=N_BANK, grid=False)
    run_bank(design, CONFIG, BANK_FOLDER, workers=WORKERS)
BANK = BANK_FOLDER / 'simulations.csv'
if not BANK.exists():
    raise FileNotFoundError('Generate the inference bank or set BANK_FOLDER to an existing bank.')
pd.read_csv(BANK).head()

## 3. Train and validate
80% of cases form the training pool; 20% are held out. By default the scaler is fitted only on the training pool. The neural network uses 10% of that pool for internal validation. The full-bank scaler option is provided for historical comparisons and includes test information. Exact paper replication also requires the original split indices and bank. Training and sampling may take a long time.

In [ ]:
from src.inference import fit_and_validate
TRAINING_SEED = 91011
SCALER_SCOPE = 'train'  # 'full' reproduces the historical scaler policy.
SPLIT_FILE = None      # Optional original scaler_and_split.npz with train/test indices.
FIT_FOLDER = OUTPUT / ('fit_' + str(TRAINING_SEED) + '_' + SCALER_SCOPE)
validation = fit_and_validate(BANK, FIT_FOLDER, mode=MODE, seed=TRAINING_SEED,
                              max_epochs=400, draws=1000, scaler_scope=SCALER_SCOPE,
                              split_file=SPLIT_FILE)
validation.head()

## 4. Synthetic validation figures
Coverage curves, parameter recovery, shrinkage and z-score, and six illustrative posteriors. The same six test cases are used for all parameters. Cases are not selected for good recovery. Small test sets give imprecise coverage estimates.

In [ ]:
validation_figures(FIT_FOLDER)

## 5. Parameter-dependent recovery accuracy and credible-interval coverage
Group synthetic test cases into equal-width intervals of their true W or G. Report MAE, RMSE, bias, 95% credible-interval coverage and counts. Error bars show pointwise 95% Wilson intervals for coverage. Empty bins remain missing. This is marginal calibration across the other parameter, not joint conditional calibration. Increase or decrease N_PARAMETER_BINS to change the intervals.

In [ ]:
from src.plotting import parameter_dependent_recovery
N_PARAMETER_BINS = 10
parameter_summary = parameter_dependent_recovery(FIT_FOLDER, bins=N_PARAMETER_BINS)
parameter_summary